In [1]:
import os
import json
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


###  Config

In [2]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE   = "../data/splits/shared_split.json"

DL_MODEL_FILE    = "../models/bilstm_model.keras"
TOKENIZER_FILE   = "../models/tokenizer.json"
LABEL_MAP_FILE   = "../models/label_mapping.json"
DL_METADATA_FILE = "../models/dl_metadata.json"
DL_PRED_FILE     = "../data/processed/dl_test_predictions.csv"
DL_SUMMARY_FILE  = "../data/processed/dl_summary.csv"

MAX_VOCAB_SIZE = 10000
MAX_LEN = 50
EMBED_DIM = 128
LSTM_UNITS = 64
BATCH_SIZE = 16
EPOCHS = 10

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

total_start_time = time.time()

### Load data

In [3]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

train_df = df[df["row_id"].isin(split_dict["train_ids"])].copy()
val_df   = df[df["row_id"].isin(split_dict["val_ids"])].copy()
test_df  = df[df["row_id"].isin(split_dict["test_ids"])].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nTotal emotion classes:", df["label"].nunique())
print("Emotion labels:", sorted(df["label"].unique().tolist()))

print("\nAll data label counts:")
print(df["label"].value_counts().sort_index())

print("\nTrain label counts:")
print(train_df["label"].value_counts().sort_index())

print("\nVal label counts:")
print(val_df["label"].value_counts().sort_index())

print("\nTest label counts:")
print(test_df["label"].value_counts().sort_index())

Train: (1512, 3)
Val  : (324, 3)
Test : (324, 3)

Total emotion classes: 6
Emotion labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']

All data label counts:
label
angry      326
disgust    427
fear       395
happy      234
neutral    500
sad        278
Name: count, dtype: int64

Train label counts:
label
angry      228
disgust    299
fear       276
happy      164
neutral    350
sad        195
Name: count, dtype: int64

Val label counts:
label
angry      49
disgust    64
fear       59
happy      35
neutral    75
sad        42
Name: count, dtype: int64

Test label counts:
label
angry      49
disgust    64
fear       60
happy      35
neutral    75
sad        41
Name: count, dtype: int64


### Tokenizer

In [4]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train = tokenizer.texts_to_sequences(train_df["text"])
X_val   = tokenizer.texts_to_sequences(val_df["text"])
X_test  = tokenizer.texts_to_sequences(test_df["text"])

X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test, maxlen=MAX_LEN, padding="post", truncating="post")

### Labels

In [5]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_val   = label_encoder.transform(val_df["label"])
y_test  = label_encoder.transform(test_df["label"])

num_classes = len(label_encoder.classes_)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_val_cat   = to_categorical(y_val, num_classes=num_classes)
y_test_cat  = to_categorical(y_test, num_classes=num_classes)

print("\nClasses:", label_encoder.classes_)
print("Number of classes:", num_classes)


Classes: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad']
Number of classes: 6


### Model

In [6]:
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS)),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

c:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train

In [7]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

train_start = time.time()

history = model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

train_elapsed = time.time() - train_start
print(f"\nTraining time: {train_elapsed:.2f} seconds")


Epoch 1/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.3644 - loss: 1.5548 - val_accuracy: 0.6235 - val_loss: 1.2153
Epoch 2/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7692 - loss: 0.7598 - val_accuracy: 0.8920 - val_loss: 0.3344
Epoch 3/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.8975 - loss: 0.2827 - val_accuracy: 0.9136 - val_loss: 0.2472
Epoch 4/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9511 - loss: 0.1648 - val_accuracy: 0.9074 - val_loss: 0.2090
Epoch 5/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9649 - loss: 0.1114 - val_accuracy: 0.8210 - val_loss: 0.4479
Epoch 6/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9636 - loss: 0.1160 - val_accuracy: 0.9043 - val_loss: 0.3053
Epoch 7/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9868 - loss: 0.0468 - val_accuracy: 0.8920 - val_loss: 0.3458

Training time: 21.50 seconds


### Evaluate

In [8]:
test_pred_prob = model.predict(X_test_pad, verbose=0)
test_pred_classes = np.argmax(test_pred_prob, axis=1)
test_pred_labels = label_encoder.inverse_transform(test_pred_classes)
test_true_labels = label_encoder.inverse_transform(y_test)

acc = accuracy_score(test_true_labels, test_pred_labels)
f1_macro = f1_score(test_true_labels, test_pred_labels, average="macro")
f1_weighted = f1_score(test_true_labels, test_pred_labels, average="weighted")

print("\n===== DL RESULT =====")
print("Accuracy    :", acc)
print("F1 Macro    :", f1_macro)
print("F1 Weighted :", f1_weighted)
print("\nClassification Report:")
print(classification_report(test_true_labels, test_pred_labels))
print("Confusion Matrix:")
print(confusion_matrix(test_true_labels, test_pred_labels))



===== DL RESULT =====
Accuracy    : 0.8981481481481481
F1 Macro    : 0.9079886526566329
F1 Weighted : 0.8956909830975488

Classification Report:
              precision    recall  f1-score   support

       angry       0.94      1.00      0.97        49
     disgust       0.90      0.94      0.92        64
        fear       0.88      0.70      0.78        60
       happy       1.00      0.97      0.99        35
     neutral       0.84      0.88      0.86        75
         sad       0.91      0.98      0.94        41

    accuracy                           0.90       324
   macro avg       0.91      0.91      0.91       324
weighted avg       0.90      0.90      0.90       324

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 60  3  0  1  0]
 [ 1  5 42  0 12  0]
 [ 0  0  0 34  0  1]
 [ 2  1  3  0 66  3]
 [ 0  1  0  0  0 40]]


### Save

In [9]:
model.save(DL_MODEL_FILE)

with open(TOKENIZER_FILE, "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())

label_mapping = {label: int(i) for i, label in enumerate(label_encoder.classes_)}
with open(LABEL_MAP_FILE, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

metadata = {
    "max_sequence_length": MAX_LEN,
    "classes": label_encoder.classes_.tolist(),
    "max_vocab_size": MAX_VOCAB_SIZE,
    "num_classes": num_classes
}
with open(DL_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

pred_df = test_df[["row_id", "text", "label"]].copy()
pred_df["dl_pred"] = test_pred_labels
pred_df["dl_confidence"] = test_pred_prob.max(axis=1)
pred_df.to_csv(DL_PRED_FILE, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame([{
    "Model": "BiLSTM",
    "Accuracy": acc,
    "F1_Macro": f1_macro,
    "F1_Weighted": f1_weighted,
    "Train_Time_Seconds": round(train_elapsed, 4),
    "Num_Classes": num_classes
}])

summary_df.to_csv(DL_SUMMARY_FILE, index=False, encoding="utf-8-sig")

print("\n===== Summary =====")
print(summary_df)

total_elapsed = time.time() - total_start_time

print("\nSaved:")
print("-", DL_MODEL_FILE)
print("-", TOKENIZER_FILE)
print("-", LABEL_MAP_FILE)
print("-", DL_METADATA_FILE)
print("-", DL_PRED_FILE)
print("-", DL_SUMMARY_FILE)
print(f"\nTotal DL pipeline time: {total_elapsed:.2f} seconds")
print("Done")


===== Summary =====
    Model  Accuracy  F1_Macro  F1_Weighted  Train_Time_Seconds  Num_Classes
0  BiLSTM  0.898148  0.907989     0.895691             21.4964            6

Saved:
- ../models/bilstm_model.keras
- ../models/tokenizer.json
- ../models/label_mapping.json
- ../models/dl_metadata.json
- ../data/processed/dl_test_predictions.csv
- ../data/processed/dl_summary.csv

Total DL pipeline time: 23.24 seconds
Done
